# Week 6 Price Right Capstone

Notebook runner for `week6_price_right_capstone.py` (plan, build, evaluate, deploy, fine-tune prep).

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from dotenv import load_dotenv

# Notebook can be opened either from repo root or from this project folder.
if Path("week6_price_right_capstone.py").exists():
    PROJECT_DIR = Path.cwd()
    REPO_ROOT = PROJECT_DIR.parents[3]
else:
    REPO_ROOT = Path.cwd()
    PROJECT_DIR = (REPO_ROOT / "week6/community-contributions/BernardUdo/week6-price-right-capstone").resolve()

SCRIPT = PROJECT_DIR / "week6_price_right_capstone.py"
VENV_PYTHON = REPO_ROOT / ".venv/Scripts/python.exe"
RUNNER_PYTHON = str(VENV_PYTHON) if VENV_PYTHON.exists() else sys.executable

# Reload .env from both repo root and this project folder.
load_dotenv(REPO_ROOT / ".env", override=True)
load_dotenv(PROJECT_DIR / ".env", override=True)

# Prefer explicit model so deploy/evaluate does not silently skip remote inference.
os.environ.setdefault("REMOTE_MODEL", "openai/gpt-4.1-mini")

print("Repo root:", REPO_ROOT)
print("Project dir:", PROJECT_DIR)
print("Script:", SCRIPT)
print("Script exists:", SCRIPT.exists())
print("Runner python:", RUNNER_PYTHON)
print("REMOTE_MODEL:", os.getenv("REMOTE_MODEL"))
print("OPENROUTER_API_KEY set:", bool(os.getenv("OPENROUTER_API_KEY")))

In [ ]:
def run_cmd(args):
    cmd = [RUNNER_PYTHON, str(SCRIPT), *args]
    print("$", " ".join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_ROOT, env=os.environ.copy())
    if out.stdout:
        print(out.stdout)
    if out.stderr:
        print(out.stderr)
    return out.returncode

## 1) Plan

In [ ]:
run_cmd(["plan"])

## 2) Build + Evaluate (Lite subset)

In [ ]:
run_cmd(["build", "--lite-mode", "--train-size", "2500", "--val-size", "500"])
run_cmd(["evaluate", "--lite-mode", "--test-size", "500"])

In [ ]:
# Re-run evaluation with explicit remote model from env.
run_cmd(["evaluate", "--lite-mode", "--test-size", "500", "--remote-model", os.getenv("REMOTE_MODEL", "openai/gpt-4.1-mini")])

## 3) Fine-tune prep + optional deploy

In [ ]:
run_cmd(["prepare-finetune", "--lite-mode", "--train-size", "100", "--val-size", "50"])
# run_cmd(["deploy", "--host", "127.0.0.1", "--port", "7866"])

## 4) Optional deploy with explicit remote model

Uncomment and run when you want the UI to call OpenRouter model predictions.

In [ ]:
# run_cmd(["deploy", "--host", "127.0.0.1", "--port", "7866", "--remote-model", os.getenv("REMOTE_MODEL", "openai/gpt-4.1-mini")])